# Agent2Agent Protocol (A2A)

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/agent-to-agent-protocol)

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

This notebook simulates the A2A **delegation** flow from scratch — an Agent Card, discovery,
and a Task moving through its lifecycle — then shows the same flow with the `a2a-sdk`.

## 1. The Agent Card

A specialist agent publishes a JSON **Agent Card** at `/.well-known/agent.json` advertising its
identity, skills, endpoint, and capabilities. Discovery = fetching this card.

In [ ]:
billing_card = {
    "name": "Billing Agent",
    "url": "http://localhost:9000/a2a",
    "capabilities": {"streaming": True},
    "skills": [{"id": "refund", "description": "Issue refunds for orders"}],
}

# an orchestrator 'discovers' the agent by reading its card
print("can it refund?", any(s["id"] == "refund" for s in billing_card["skills"]))

## 2. The Task lifecycle

A Task moves `submitted → working → completed` (or asks for more input, or fails). The remote
agent streams these states, and the final message carries an **Artifact** — the produced output.

In [ ]:
def billing_agent(user_text):
    """Yield JSON-RPC status updates, ending with an Artifact."""
    yield {"status": {"state": "submitted"}}
    yield {"status": {"state": "working"}}
    order = user_text.split()[-1]
    yield {"status": {"state": "completed"},
           "artifacts": [{"parts": [{"kind": "text",
               "text": f"Refunded {order} (txn r_88c2)"}]}]}

## 3. The orchestrator delegates

The client sends one Message (made of Parts) and consumes the streamed states until `completed`,
then reads the Artifact. Note it handles every state, not just the last one.

In [ ]:
message = {"role": "user", "parts": [{"kind": "text", "text": "Refund order #4471"}]}

artifact = None
for update in billing_agent(message["parts"][0]["text"]):
    state = update["status"]["state"]
    print("state:", state)
    if state == "completed":
        artifact = update["artifacts"][0]["parts"][0]["text"]

print("artifact:", artifact)

## 4. The same flow with the a2a-sdk

The `a2a-sdk` serves the card at the well-known URL and manages the Task lifecycle for you.
Run the server with `uvicorn billing_agent:app --port 9000`, then discover + delegate from a client.

```bash
pip install a2a-sdk uvicorn httpx
```

In [ ]:
client_code = '''
import asyncio, httpx
from a2a.client import A2ACardResolver, A2AClient
from a2a.types import MessageSendParams, SendMessageRequest

async def main():
    async with httpx.AsyncClient() as http:
        card = await A2ACardResolver(http, "http://localhost:9000").get_agent_card()
        print(card.skills[0].id)          # discovered, not hard-coded
        client = A2AClient(http, agent_card=card)
        req = SendMessageRequest(params=MessageSendParams(
            message={"role": "user",
                     "parts": [{"kind": "text", "text": "Refund order #4471"}]}))
        print(await client.send_message(req))

asyncio.run(main())
'''
print(client_code)

## ✏️ Your turn

Make the billing agent emit an **`input-required`** state before completing — asking which card to
refund to — and have the orchestrator answer it before the task can reach `completed`.

In [ ]:
def billing_agent_v2(user_text, answer=None):
    yield {"status": {"state": "working"}}
    # TODO(you): if answer is None, yield an 'input-required' state and stop
    # otherwise, complete using `answer`
    ...

# first pass should pause on input-required
states = [u["status"]["state"] for u in billing_agent_v2("Refund order #4471")]
assert "input-required" in states
print("states:", states)

<details>
<summary>Solution</summary>

```python
def billing_agent_v2(user_text, answer=None):
    yield {"status": {"state": "working"}}
    if answer is None:
        yield {"status": {"state": "input-required"},
               "prompt": "Which card should I refund to?"}
        return
    order = user_text.split()[-1]
    yield {"status": {"state": "completed"},
           "artifacts": [{"parts": [{"kind": "text",
               "text": f"Refunded {order} to {answer}"}]}]}
```
</details>